# Training both branches on synthetic data

This notebook demonstrates the full training flow for the RNN regression and
CNN classification branches. It runs on CPU with no network access, using the
built-in synthetic klines generator. For real data use the CLI:

```bash
volnn download && volnn prepare --model rnn && volnn train --model rnn
```

In [ ]:
import numpy as np
from volnn.config import Config, DataConfig, FeatureConfig, TrainConfig
from volnn.data.prepare import prepare
from volnn.data.synthetic import write_klines_csv
from volnn.train import train_model, evaluate_model

PAST, FUTURE, EPOCHS = 48, 6, 3
klines_path = write_klines_csv("../data/NB-5m.csv", n_steps=6000, seed=42)
features = FeatureConfig(ticker="NB", interval="5m", past=PAST, future=FUTURE, threshold_5m=0.05)

## RNN regression branch

In [ ]:
rnn_cfg = Config(
    data=DataConfig(ticker="NB", interval="5m"),
    features=features,
    train=TrainConfig(model="rnn", epochs=EPOCHS, batch_size=128, seed=42),
)
prepare(rnn_cfg, klines_path)
rnn_history = train_model(rnn_cfg)
rnn_metrics = evaluate_model(rnn_cfg)
rnn_metrics

## CNN classification branch

In [ ]:
cnn_cfg = Config(
    data=DataConfig(ticker="NB", interval="5m"),
    features=features,
    train=TrainConfig(model="cnn", epochs=EPOCHS, batch_size=128, seed=42),
)
prepare(cnn_cfg, klines_path)
cnn_history = train_model(cnn_cfg)
cnn_metrics = evaluate_model(cnn_cfg)
cnn_metrics

## Compare with baselines

The learned models are only interesting if they beat simple baselines.

In [ ]:
from volnn.baselines import majority_baseline, persistence_baseline
from volnn.data.labels import build_regression_dataset, build_classification_dataset
from volnn.data.features import build_feature_frame
import pandas as pd

klines = pd.read_csv(klines_path, header=None)
frame = build_feature_frame(klines, features)
hv = frame[:, 0]

reg = build_regression_dataset(frame, hv, PAST, FUTURE)
naive = persistence_baseline(hv, PAST, FUTURE)
print("naive persistence MAE:", float(np.mean(np.abs(naive - reg.y))))

clf = build_classification_dataset(frame, hv, PAST, FUTURE, features.threshold())
maj = majority_baseline(clf.y)
print("majority-class accuracy:", float(np.mean(maj == clf.y)))